# tl;dr

The checkpoint-anchored Gompertz candidate reached **124.8 g held-out MAE**, compared with **127.1 g** for historical remaining gain. The 2.3 g pooled difference is too small to be operationally meaningful, and a paired cycle bootstrap includes no improvement. On the untouched 2026-3 audit, Gompertz was slightly weaker (**84.6 g versus 77.6 g**). The checkpoint model should therefore remain the provisional default; Gompertz is a research comparison only.

When the current weight is blank, the Gompertz curve returns the same population-level Day 35 estimate for every building. That is a generic prior, not a building-specific forecast.

## Context & Methods

**Question.** Can a Gompertz growth curve improve the Day 35 bodyweight outlook, including when a checkpoint weight is missing?

**Design.** The experiment uses 31 development building-cycles from 2025-2 through 2026-2. Each outer fold holds out one complete production cycle. The three 2026-3 buildings remain untouched until the method is frozen. The candidate fits a robust population Gompertz curve on training weights from Days 7, 14, 21, 28, and 35.

With a measured checkpoint weight, the candidate preserves the building's current deviation and adds the curve-implied remaining gain. Without a measured weight, only the population curve endpoint is available.

In [1]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "canary").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.evaluate_gompertz_candidate import evaluate

OUTPUT = ROOT / "analysis" / "gompertz_evaluation"
results = evaluate(OUTPUT)
print(f"Source: {results['source']}")
print(f"Development building-cycles: {results['development_building_cycles']}")
print(f"Untouched audit: {results['audit_buildings']} buildings from {results['audit_cycle']}")

Source: /Users/jourdan.go/Downloads/PROJECT CANARY/Project_Canary_GitHub_Ready/data/FARM HARVEST DATA.xlsx
Development building-cycles: 31
Untouched audit: 3 buildings from 2026-3


## Data

The source is the canonical farm workbook. The target is recorded average Day 35 bodyweight. The candidate uses only bodyweights available within each training fold; 2026-3 does not participate in fitting or model choice.

In [2]:
assert results["development_building_cycles"] == 31
assert results["audit_cycle"] == "2026-3"
assert results["audit_buildings"] == 3
assert results["development_metrics"]["rows"] == 124
assert all(item["rows"] == 31 for item in results["checkpoint_metrics"])
print("Cohort and checkpoint assertions passed.")

Cohort and checkpoint assertions passed.


## Results

In [3]:
checkpoint = results["existing_models"]["checkpoint_bodyweight"]
checkpoint_audit = results["existing_models"]["checkpoint_bodyweight_audit"]
model3 = results["existing_models"]["model_3_day21_bodyweight"]
model3_audit = results["existing_models"]["model_3_day21_bodyweight_audit"]
gompertz = results["development_metrics"]
gompertz_audit = results["audit_metrics"]

comparison = pd.DataFrame([
    {"Engine": "Model 3 — XGBoost", "Forecast availability": "Day 21 only", "Development MAE": f"{model3['mae']:.1f} g", "2026-3 audit MAE": f"{model3_audit['mae']:.1f} g"},
    {"Engine": "Checkpoint — historical remaining gain", "Forecast availability": "Days 7, 14, 21, 28", "Development MAE": f"{checkpoint['mae_g']:.1f} g", "2026-3 audit MAE": f"{checkpoint_audit['mae_g']:.1f} g"},
    {"Engine": "Gompertz — anchored remaining gain", "Forecast availability": "Days 7, 14, 21, 28", "Development MAE": f"{gompertz['mae_g']:.1f} g", "2026-3 audit MAE": f"{gompertz_audit['mae_g']:.1f} g"},
])
comparison

,Engine,Forecast availability,Development MAE,2026-3 audit MAE
0,Model 3 — XGBoost,Day 21 only,146.2 g,116.5 g
1,Checkpoint — historical remaining gain,"Days 7, 14, 21, 28",127.1 g,77.6 g
2,Gompertz — anchored remaining gain,"Days 7, 14, 21, 28",124.8 g,84.6 g


In [4]:
gompertz_by_day = pd.DataFrame(results["checkpoint_metrics"])[["review_day", "mae_g"]].rename(columns={"mae_g": "Gompertz MAE (g)"})
checkpoint_by_day = pd.DataFrame(results["existing_models"]["checkpoint_bodyweight"].get("checkpoint_metrics", []))
stored_checkpoint = pd.read_csv(ROOT / "models" / "three_model" / "checkpoint_champion" / "checkpoint_metrics.csv")
stored_checkpoint = stored_checkpoint[["review_day", "mae_g"]].rename(columns={"mae_g": "Checkpoint MAE (g)"})
gompertz_by_day.merge(stored_checkpoint, on="review_day")

,review_day,Gompertz MAE (g),Checkpoint MAE (g)
0,7,151.432313,155.450495
1,14,137.629886,138.060996
2,21,109.722266,112.200292
3,28,100.547178,102.771816


In [5]:
paired = results["paired_comparison_with_checkpoint"]
pd.Series({
    "Cycle-macro MAE difference, Gompertz minus checkpoint (g)": paired["point_difference_g"],
    "Paired bootstrap 95% low (g)": paired["ci95_low_g"],
    "Paired bootstrap 95% high (g)": paired["ci95_high_g"],
    "Bootstrap probability Gompertz is lower-error": paired["probability_gompertz_lower_mae"],
})

Cycle-macro MAE difference, Gompertz minus checkpoint (g)   -1.667099
Paired bootstrap 95% low (g)                                -8.870014
Paired bootstrap 95% high (g)                                3.018240
Bootstrap probability Gompertz is lower-error                0.668600
dtype: float64

In [6]:
pd.Series(results["blank_weight_fallback"])

interpretation                       Generic cohort prior only; it is identical for...
gompertz_development_mae_g                                                  157.985336
historical_mean_development_mae_g                                            163.03632
gompertz_audit_mae_g                                                         60.681888
historical_mean_audit_mae_g                                                  43.002581
dtype: object

## Takeaways

1. The anchored Gompertz candidate is numerically close to the checkpoint method, not decisively better.
2. Its small development advantage is within paired uncertainty, and it is weaker on the later 2026-3 audit.
3. The fitted mature-weight asymptote is weakly identified because the records end at Day 35; one outer fold reaches the allowed upper bound. This makes biological interpretation of the parameters unsafe.
4. A blank checkpoint weight removes the building-specific anchor. The output then becomes a generic cohort estimate and should be labelled as such, not shown as a normal building forecast.
5. Do not change the Canary app based on this experiment. Retain historical remaining gain as the provisional checkpoint model and keep Gompertz available only for research comparison.